In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# constructor year and driver year pairs
# mean points per year dependent
# Avg pit stop duration
# nationality
# fastest lap position
# normalized lap standard deviation
# region: UK, Europe, Asia, Americas, Oceania, Africa

In [3]:
constructors = pd.read_csv('f1_data/constructors.csv')
constructor_standings = pd.read_csv('f1_data/constructor_standings.csv')
constructor_results = pd.read_csv('f1_data/constructor_results.csv')
driver_standings = pd.read_csv('f1_data/driver_standings.csv')
drivers = pd.read_csv('f1_data/drivers.csv')
lap_times = pd.read_csv('f1_data/lap_times.csv')
pit_stops = pd.read_csv('f1_data/pit_stops.csv')
results = pd.read_csv('f1_data/results.csv')
races = pd.read_csv('f1_data/races.csv')


In [4]:
driver_standings_w_years = pd.merge(driver_standings, races[['raceId', 'year']], on='raceId')
driver_standings_summary = driver_standings_w_years.groupby(['driverId', 'year'], as_index=False).agg(mean_points = ('points', 'mean'))

constructor_standings_w_years = pd.merge(constructor_standings, races[['raceId', 'year']], on='raceId')
constructor_standings_summary = constructor_standings_w_years.groupby(['constructorId', 'year'], as_index=False).agg(mean_points=('points', 'mean'))

In [5]:
pit_stops_w_years = pd.merge(pit_stops, races[['raceId', 'year']], on='raceId')
pit_stops_w_constructors = pd.merge(pit_stops_w_years, results[['raceId', 'driverId', 'constructorId']], on=['raceId', 'driverId'])
pit_stops_w_constructors['duration'] = pit_stops_w_constructors['milliseconds'] * 10**-3

driver_pit_stops_summary = pit_stops_w_constructors.groupby(['driverId', 'year'], as_index=False).agg(avg_pit_stop_duration = ('duration','mean'))
constructor_pit_stops_summary = pit_stops_w_constructors.groupby(['constructorId', 'year'], as_index=False).agg(avg_pit_stop_duration = ('duration','mean'))

In [6]:
lap_times['duration'] = lap_times['milliseconds'] * 10**-3

avg_lap_times = lap_times.groupby('raceId').agg(avg_lap = ('duration', 'mean'))
lap_times_w_avg = pd.merge(lap_times, avg_lap_times, on='raceId')
lap_times_w_avg['duration_standard'] = lap_times_w_avg['duration'] / lap_times_w_avg['avg_lap']

lap_times_w_years = pd.merge(lap_times_w_avg, races[['raceId', 'year']], on='raceId')
lap_times_w_constructors = pd.merge(lap_times_w_years, results[['raceId', 'driverId', 'constructorId']], on=['raceId', 'driverId'])

driver_lap_times_summary = lap_times_w_constructors.groupby(['driverId', 'year'], as_index=False).agg(stdev_lap_time = ('duration_standard', 'std'))
constructor_lap_times_summary = lap_times_w_constructors.groupby(['constructorId', 'year'], as_index=False).agg(stdev_lap_time = ('duration_standard', 'std'))

In [11]:
results['fastestLap'] = pd.to_numeric(results['fastestLap'].replace(r'\N', np.nan))
results['fastest_lap_pos'] = results['fastestLap'] / results['laps']

results_w_years = pd.merge(results, races[['raceId', 'year']], on='raceId')
driver_fastest_lap_summary = results_w_years.groupby(['driverId', 'year'], as_index=False).agg(fastest_lap = ('fastest_lap_pos', 'mean'))
constructor_fastest_lap_summary = results_w_years.groupby(['constructorId', 'year'], as_index=False).agg(fastest_lap = ('fastest_lap_pos', 'mean'))

In [12]:
drivers_df = pd.merge(drivers[['driverId', 'forename', 'surname', 'nationality']], driver_standings_summary, on='driverId')
drivers_df = pd.merge(drivers_df, driver_pit_stops_summary, on=['driverId', 'year'])
drivers_df = pd.merge(drivers_df, driver_lap_times_summary, on=['driverId', 'year'])
drivers_df = pd.merge(drivers_df, driver_fastest_lap_summary, on=['driverId', 'year'])

In [14]:
constructors_df = pd.merge(constructors[['constructorId', 'nationality']], constructor_standings_summary, on='constructorId')
constructors_df = pd.merge(constructors_df, constructor_pit_stops_summary, on=['constructorId', 'year'])
constructors_df = pd.merge(constructors_df, constructor_lap_times_summary, on=['constructorId', 'year'])
constructors_df = pd.merge(constructors_df, constructor_fastest_lap_summary, on=['constructorId', 'year'])